# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dawngend/FlyRank-Machine-Learning-Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### The lane, restated as a question shape

The Freestyle lane asks **"which pages should a human refresh first?"** That is a ranking question,
not a yes/no question, even though the label (`is_declining_label`) is binary. `skills/training-honest-models`
maps that shape explicitly: *"which first?" ranking -> any classifier's probability, evaluated at
precision@K, because ranking needs scores, not labels.*

So every method below is judged on its **predicted probability**, and the headline metric is
precision@K. Accuracy is not reported anywhere in this notebook, because a queue is never consumed by
thresholding at 0.5 — it is consumed from the top down until the reviewer runs out of time.

### The three methods, and why each is here

| Method | Why it is in the comparison |
|---|---|
| **Logistic Regression** | The readable starting point the skill names first. Linear, inspectable coefficients, and if it matches the forest then the extra complexity is not earned. |
| **Random Forest** | The "stronger" half of the skill's readable -> stronger progression. Handles the non-linear thresholds the ML-07 rule encoded by hand (`word_count < 1200`, `avg_position <= 10`) without me choosing the cut points. |
| **Decision tree, depth 2** | Included purely to be *printed*. The skill's position is that "a depth-2 decision tree you can print and read teaches more than an opaque model 2 points stronger". It is not expected to win, and it does not. It is here so section 4 can show an actual decision surface instead of describing one. |

The methods and the split below are recorded decisions rather than defaults arrived at in code, and
`work/scripts/train_refresh_model.py` states them at the top of the file for the same reason.

### What is deliberately not here

No gradient boosting, no neural model, no hyperparameter search. ML-07 established that the signal is
weak and that the data has real quality problems (52 of its top 100 rows had `word_count` missing).
Tuning a stronger model against a corpus with known defects optimises against the defects. The
comparison that matters at this stage is rule versus learned model, not model versus model.

**Seed: 42**, fixed in `train_refresh_model.py` and used for every split and every estimator.

In [1]:
# Section 1 - resolve the repo, load the contract, restate the method decisions.
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)


def find_repo_root() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents, Path("/content/FlyRank-Machine-Learning-Internship")]:
        if (base / "work" / "outputs" / "data_contract.json").exists():
            return base
    raise FileNotFoundError(
        "Could not locate the repo root. In Colab, clone the repo first:\n"
        "  !git clone https://github.com/Dawngend/FlyRank-Machine-Learning-Internship.git"
    )


ROOT = find_repo_root()
sys.path.insert(0, str(ROOT / "work" / "scripts"))

from train_refresh_model import K_VALUES, N_SPLITS, SEED

CONTRACT = json.loads((ROOT / "work" / "outputs" / "data_contract.json").read_text(encoding="utf-8"))
frame = pd.read_csv(ROOT / "data" / "processed" / "refresh_feature_vector.csv")

print(f"repo root  : {ROOT}")
print(f"contract   : v{CONTRACT['version']}  decision date T = {CONTRACT['windows']['decision_date']}")
print(f"rows       : {len(frame):,}")
print(f"seed       : {SEED}")
print(f"metric     : precision@K for K in {K_VALUES}  (ranking question, not a 0.5 threshold)")
print(f"methods    : logistic_regression -> random_forest, plus decision_tree_depth2 (printed, not competing)")

repo root  : D:\Flyrank\FlyRank-Machine-Learning-Internship
contract   : v1.1  decision date T = 2026-03-31
rows       : 30,000
seed       : 42
metric     : precision@K for K in (20, 50, 100, 500, 1000)  (ranking question, not a 0.5 threshold)
methods    : logistic_regression -> random_forest, plus decision_tree_depth2 (printed, not competing)


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### The decision: grouped by `client_id`, 5-fold

**No client appears in both train and test.** The split is `GroupKFold(n_splits=5)` on `client_id`.

### Why a random split would not be honest here

This is not a stylistic preference; the corpus makes it a correctness issue, and the next cell
measures it. There are only **32 clients** across 30,000 rows, they are severely imbalanced (3 rows at
the smallest, 7,008 at the largest, so a single client is 23% of the corpus), and **the declining rate
per client ranges from 0.000 to 0.937** against a corpus base rate of 0.542.

That spread is the problem. Under a random split the same client sits on both sides, and a model can
score well by inferring which client a row belongs to from its feature fingerprint and predicting that
client's base rate. It would look like a content model and behave like a client lookup table. The
resulting number would be real but would not answer the lane's question, because in use the queue is
run for a client whose pages the model has not seen.

Grouping by client removes that route. The cost is honest and worth stating: with 32 groups and one
dominant client, fold composition varies a lot, so **every metric in section 3 is reported as a mean
across folds with its standard deviation**, and those deviations are large. A single-number score would
misrepresent the certainty available from this corpus.

### Why the split is not time-aware, and what would make it so

The ML-04 contract specifies a sealed test month (`2026-06`) and an iteration partition
(`month=2026-03`). **Neither can be applied to the starter CSV**, because it is a single undated
trailing-90-day snapshot: the only date-like columns are durations (`content_age_days`,
`days_since_last_update`, `days_with_impressions`, `days_with_sessions`), not observation dates. The
next cell asserts that no true date column exists rather than asking the reader to take it on trust.

So the sealed-month design is recorded here as **the split to use once dated warehouse data is
available**, and grouping by client is the strongest honest split obtainable today. When the warehouse
fact table is in play, the correct design is grouped *and* time-aware: group by client, train on
`month <= 2026-03`, and never touch `2026-06` until the model is final.

In [2]:
# Section 2 - verify the split is necessary, and that a time-aware split is not available.
groups = frame["client_id"]
y = frame["is_declining_label"]

per_client = frame.groupby("client_id")["is_declining_label"].agg(["size", "mean"]).sort_values("size", ascending=False)
print(f"clients            : {per_client.shape[0]}")
print(f"rows per client    : min {per_client['size'].min():,}  median {int(per_client['size'].median()):,}  max {per_client['size'].max():,}")
print(f"largest client     : {per_client['size'].max() / len(frame):.1%} of the corpus")
print(f"corpus base rate   : {y.mean():.3f}")
print(f"per-client rate    : min {per_client['mean'].min():.3f}  max {per_client['mean'].max():.3f}  spread {per_client['mean'].max() - per_client['mean'].min():.3f}")

print("\nTop 8 clients by size (size, declining rate):")
print(per_client.head(8).to_string())

# The claim that a time-aware split is unavailable, asserted rather than asserted-in-prose.
DURATION_COLS = {"days_with_impressions", "days_with_sessions", "content_age_days", "days_since_last_update"}
date_like = [c for c in frame.columns if any(k in c.lower() for k in ("date", "month", "timestamp"))]
true_dates = [c for c in date_like if c not in DURATION_COLS]
assert not true_dates, f"a real date column exists after all: {true_dates}"
print(f"\n[OK] no observation-date column in the starter CSV -> time-aware split is not available")
print(f"[OK] contract sealed test month {CONTRACT['windows']['sealed_test_month']} recorded as future design, not applied here")
print(f"[OK] split = GroupKFold(n_splits={N_SPLITS}) on client_id")

clients            : 32
rows per client    : min 3  median 567  max 7,008
largest client     : 23.4% of the corpus
corpus base rate   : 0.542
per-client rate    : min 0.000  max 0.937  spread 0.937

Top 8 clients by size (size, declining rate):
                   size      mean
client_id                        
client_19581e27de  7008  0.490154
client_6208ef0f77  3681  0.643847
client_4e07408562  2294  0.493461
client_3fdba35f04  2267  0.838553
client_f369cb89fc  1796  0.601336
client_8527a891e2  1194  0.437186
client_a88a7902cb  1171  0.667805
client_d4735e3a26  1106  0.219711

[OK] no observation-date column in the starter CSV -> time-aware split is not available
[OK] contract sealed test month 2026-06 recorded as future design, not applied here
[OK] split = GroupKFold(n_splits=5) on client_id


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

The ML-07 rule is **imported** from `work/scripts/baseline_action_score.py` and scored on the identical
test rows of each fold, rather than compared against the single-number result quoted in the Week-4
notebook. This matters: the Week-4 figures were computed over all 30,000 rows at once, and the numbers
below are computed fold by fold on held-out clients, so the baseline's own numbers move. Comparing the
new models against the old whole-corpus figure would have flattered them.

In [3]:
# Section 3 - the comparison table. Same rows, same folds, same metric for every method.
from train_refresh_model import run

RESULTS_PATH = ROOT / "work" / "outputs" / "model_comparison.json"
summary = json.loads(RESULTS_PATH.read_text(encoding="utf-8")) if RESULTS_PATH.exists() else run()

base_rate = summary["base_rate"]
metric_keys = ["roc_auc", "average_precision"] + [f"p@{k}" for k in K_VALUES]

rows = []
for name, block in summary["metrics"].items():
    row = {"model": name}
    for m in metric_keys:
        mean, std = block[m]["mean"], block[m]["std"]
        row[m] = f"{mean:.3f} +-{std:.2f}" if mean is not None else "n/a"
    rows.append(row)

table = pd.DataFrame(rows).set_index("model")
print(f"Base rate: {base_rate:.3f}   |   split: {summary['split']['type']}, "
      f"{summary['split']['n_splits']} folds over {summary['split']['n_clients']} clients")
print(f"Every cell is mean +- std across folds.\n")
print(table.to_string())

# Lift over the base rate, which is the number that says whether the queue is worth running.
print(f"\nLift over base rate ({base_rate:.3f}):")
lift = pd.DataFrame({
    m: {name: (summary["metrics"][name][m]["mean"] / base_rate)
        for name in summary["metrics"]}
    for m in [f"p@{k}" for k in K_VALUES]
})
print(lift.round(2).to_string())

Base rate: 0.542   |   split: GroupKFold on client_id, 5 folds over 32 clients
Every cell is mean +- std across folds.

                           roc_auc average_precision          p@20          p@50         p@100         p@500        p@1000
model                                                                                                                     
baseline_rule         0.539 +-0.03      0.568 +-0.09  0.610 +-0.20  0.616 +-0.14  0.602 +-0.17  0.592 +-0.11  0.558 +-0.07
logistic_regression   0.660 +-0.04      0.667 +-0.09  0.780 +-0.16  0.788 +-0.10  0.752 +-0.11  0.728 +-0.08  0.710 +-0.08
random_forest         0.671 +-0.04      0.680 +-0.05  0.780 +-0.11  0.776 +-0.09  0.776 +-0.08  0.751 +-0.05  0.726 +-0.05
decision_tree_depth2  0.608 +-0.06      0.606 +-0.08  0.640 +-0.04  0.628 +-0.08  0.608 +-0.06  0.612 +-0.07  0.613 +-0.07

Lift over base rate (0.542):
                      p@20  p@50  p@100  p@500  p@1000
baseline_rule         1.13  1.14   1.11   1.09    1.03
lo

### Reading the table

**The learned models beat the rule, and they beat it where the rule actually failed.** ML-07's central
weakness was depth: its lift collapsed to 1.01x by k=500 and 0.99x by k=1,000, so it could only order
the first hundred-odd rows. Under the same grouped folds the baseline reaches p@1000 = 0.558 against a
0.542 base rate, essentially the same collapse. The random forest holds **p@1000 = 0.726**, a 1.34x
lift sustained a thousand rows deep. That is the difference between "a way to find 50 pages" and "a
queue you can actually work through", and it is the single result that justifies moving off the rule.

**Logistic regression is not meaningfully worse than the forest at the top.** Both reach p@20 = 0.780.
The forest's advantage appears only at depth (p@500 0.751 vs 0.728, p@1000 0.726 vs 0.710). Following
the skill's instruction to report the split decision rather than the winner: *if the readable model
matches the ensemble in the region a human actually reads, the readable model is the better
deliverable.* If this queue is only ever worked 100 rows deep, logistic regression is the honest choice
and the forest is unnecessary complexity.

**The fold deviations are large and they dominate several comparisons.** The baseline's p@20 carries
+-0.20 and the two strong models +-0.11 to +-0.16. Against that, the p@20 difference between logistic
regression and the forest (0.780 vs 0.780) is not a difference at all, and even several of the gaps at
p@100 sit inside one standard deviation. With 32 clients and one of them holding 23% of the rows, this
is the precision the corpus supports. Any claim of the form "model A beats model B by N points" that
does not clear these deviations is not supported here, and none is made.

**ROC-AUC and precision@K disagree, and precision@K is the one that counts.** The forest's ROC-AUC is
0.671, which is modest. Its p@100 is 0.776, which is strong. Both are true: the model is mediocre at
separating the full corpus and good at identifying the top of it. For a refresh queue, only the second
property is used, which is exactly why the lane's metric was fixed as precision@K in section 1 rather
than chosen after seeing results.

**The depth-2 tree loses, as expected**, at ROC-AUC 0.608 and p@100 0.608. It stays in the comparison
because section 4 prints it.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [4]:
# Section 4 - what the model leans on, and where it is wrong.

# --- 4a. The depth-2 tree, printed --------------------------------------------
print("Depth-2 decision tree (fold 1). Thresholds are on standardised features,")
print("so a negative cut means 'below average'.\n")
print(summary["decision_tree_depth2_rules"])

# --- 4b. What the forest leans on ---------------------------------------------
imp = pd.Series(summary["random_forest_top_features"]).sort_values(ascending=False)
print("\nRandom forest, mean feature importance across folds (top 15):")
for feat, val in imp.items():
    print(f"  {feat:<34} {val:.4f}  {'#' * int(val * 200)}")

# --- 4c. Cross-check against what the ML-07 rule believed ---------------------
print("\nWhat ML-07 weighted most, vs what the forest actually uses:")
ml07_weighting = {
    "days_since_last_update": ("stale_visible_page", 30),
    "word_count": ("thin_visible_page", 25),
    "avg_position": ("page_one_decay_risk", 20),
    "ctr": ("low_ctr_visible_page", 15),
    "engagement_rate": ("low_engagement_visible_page", 10),
}
ranked = {f.replace("num__", "").replace("cat__", ""): i + 1 for i, f in enumerate(imp.index)}
for feat, (code, pts) in ml07_weighting.items():
    rank = ranked.get(feat)
    val = imp.get(f"num__{feat}")
    rank_s = f"#{rank}" if rank else "outside top 15"
    val_s = f"{val:.4f}" if val is not None else "n/a"
    print(f"  {feat:<24} ML-07 gave it {pts:>2} pts ({code:<28}) -> forest rank {rank_s:<15} imp {val_s}")

Depth-2 decision tree (fold 1). Thresholds are on standardised features,
so a negative cut means 'below average'.

|--- num__days_with_impressions <= -1.499
|   |--- num__avg_position <= -1.034
|   |   |--- class: 0
|   |--- num__avg_position >  -1.034
|   |   |--- class: 0
|--- num__days_with_impressions >  -1.499
|   |--- num__content_age_days <= 0.881
|   |   |--- class: 1
|   |--- num__content_age_days >  0.881
|   |   |--- class: 0


Random forest, mean feature importance across folds (top 15):
  num__days_with_impressions         0.1302  ##########################
  num__log_impressions_90d           0.1296  #########################
  num__avg_position                  0.1084  #####################
  num__content_age_days              0.0835  ################
  num__word_count                    0.0457  #########
  num__char_count                    0.0449  ########
  num__ctr                           0.0402  ########
  num__log_clicks_90d                0.0353  #######
  num__

### What the model leans on, and what that says about ML-07

**The forest's top four features are all about observed visibility, not about content.**
`days_with_impressions` (0.130), `log_impressions_90d` (0.130), `avg_position` (0.108) and
`content_age_days` (0.084) together account for roughly 45% of total importance. Content shape
(`word_count` 0.046, `char_count` 0.045) sits well behind them.

**The signal ML-07 weighted highest is nearly worthless to the model.** `days_since_last_update`
carried 30 points in the rule, the largest single weight, on the reasoning that a page nobody has
touched in six months is the most urgent case. The forest ranks it **12th at 0.023 importance**, below
`scroll_rate`. Two independent lines of evidence now agree that the staleness code was a bad bet: it
fired on 17 of 30,000 rows in ML-07, and the model does not use it when it is free to. Recency of
editing appears not to carry information about whether a page is declining, at least in this corpus.

**The depth-2 tree makes the same point in one readable picture.** Its first split is on
`days_with_impressions`, and pages appearing on few days are classified as not declining regardless of
what the second split says (both leaves under that branch are class 0). Its second split, for pages
that do appear consistently, is `content_age_days`. So the tree's entire learned rule is roughly *"a
page that shows up consistently and is not yet very old is the one at risk"* — a statement about
sustained visibility and age, containing nothing about word count, CTR, or edit recency. It is a weak
model (p@100 = 0.608), but it is a legible one, and it agrees with the forest about where to look.

**Where the model is wrong.** Its ROC-AUC of 0.671 means it separates the full corpus only modestly;
roughly a third of pairwise orderings are wrong. The fold deviations show where that concentrates: the
metrics swing by up to +-0.20 depending on which clients land in the test fold, so performance is
markedly client-dependent, and there is no basis here for claiming it will generalise evenly to a new
client. The dominant client holding 23% of rows makes this worse, since folds containing it are not
comparable to folds that do not.

### Claims, stated at the strength the evidence supports

- **Observed:** under a client-grouped 5-fold split, a random forest ranked declining pages with
  precision@100 of 0.776 +-0.08 against a base rate of 0.542.
- **Observed:** the same split scored the ML-07 rule at precision@100 of 0.602 +-0.17.
- **Directional:** sustained visibility signals appear more informative than content-shape or
  edit-recency signals for this outcome.
- **Decision-support, not causal:** nothing here shows that refreshing a highly-ranked page changes its
  trend. The model orders pages by how likely they are to be *observed* declining, which is a
  prioritisation aid for a human reviewer and not evidence that the recommended action works.

### What ML-09 has to check

The precision@K figures are strong enough to be worth doubting. ML-09 re-runs this model under its own
honest-split audit and repeats the Week-3 leakage hunt on this exact feature set, because a jump from
0.602 to 0.776 is the size of gap that leakage usually explains.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.